In [29]:
# 导入必要的库
import numpy as np                          # 数值计算库
from vpsto.vpsto import VPSTO, VPSTOOptions # VPSTO轨迹优化库
from vpsto.obf import OBF                   # 椭球基函数库
import matplotlib.pyplot as plt             # 绘图库
import matplotlib.patches as patches        # matplotlib几何图形补丁

# Jupyter notebook魔法命令，用于自动重新加载模块
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# 定义2D 3自由度 机械臂类，包含正向运动学计算
class Manipulator():
    def __init__(self):
        # 机械臂链接长度：长度均为1
        self.l = np.array([1, 1, 1]) # link lengths
        # 关节角度限制：第一关节[0, π]，第二关节[-π, 0]
        self.q_min = np.array([0., -np.pi, -np.pi])    # 关节角度下限
        self.q_max = np.array([np.pi, 0., np.pi])     # 关节角度上限

    # 定义机械臂的正向运动学
    def fk(self, q):
        """
        正向运动学函数：根据关节角度计算机械臂各关节的笛卡尔坐标位置
        参数：
        q: 2x1数组，包含所有关节的角度
        返回：(N+1，2)矩阵，每行都是一个二维坐标位置，包含基座、所有关节和末端执行器的二维坐标位置
        """
        x0 = np.zeros(2)  # 基座位置（原点）
        # 第一关节位置：基座 + 第一段链接的笛卡尔位置
        x1 = x0 + self.l[0] * np.array([np.cos(q[0]), np.sin(q[0])])
        # ：第一关节 + 第二段链接的笛卡尔位置
        x2 = x1 + self.l[1] * np.array([np.cos(q[0] + q[1]), np.sin(q[0] + q[1])])
        # 末端执行器位置: 第二关节 + 第三段链接的笛卡尔位置
        x3 = x2 + self.l[2] * np.array([np.cos(q[0] + q[1] + q[2]), np.sin(q[0] + q[1] + q[2])])
        # 垂直堆栈列向量
        return np.vstack((x0, x1, x2, x3))  # 返回所有关节的位置

# 创建机械臂实例并测试正向运动学
# mp = Manipulator()  # 创建机械臂实例
# q = np.array([np.pi/4, -np.pi/2, np.pi/4])  # 定义关节角度
# positions = mp.fk(q)  # 计算正向运动学，获取各关节位置
# print("关节位置：\n", positions)  # 打印关节位置
# print("pos维度：", positions.shape)  # 打印位置数组的维度


In [31]:
'''
Author: Fang Kai[thissfk@qq.com]
Date: 2025-09
LastEditors: Fang Kai[thissfk@qq.com]
LastEditTime: 2025-09
FilePath: 2D_3dof_arm_av_collision.ipynb
Description: 
           If you need more information,
please contact Fang Kai[thissfk@qq.com] to get an access.   
Copyright (c) 2025 by Fang Kai, All Rights Reserved. 
'''
is_tested = 0
# 碰撞环境类：包含球形障碍物和工作空间边界
class CollisionEnvironment():
    def __init__(self):
        # 球形障碍物的中心位置
        # self.x = np.array([0.5, 0.5])
        # !!!test
        self.x = np.array([1, 1.5])
        # 球形障碍物的半径
        self.r = 0.1
        # 球形障碍物半径的平方（用于优化碰撞检测计算）
        self.r_sq = self.r**2

        # 工作空间的边界限制
        self.x_min = np.array([-0.75, 0.])     # 工作空间最小边界
        self.x_max = np.array([1.5, 1.5])     # 工作空间最大边界

    def isCollision(self, pts):
        """
        检查线段是否与球形障碍物发生碰撞
        参数：
        pts: (n, 4)数组，每行包含一条线段的两个二维端点坐标
        返回：布尔数组，表示每条线段是否与障碍物碰撞
        """
        # 计算线段方向向量
        e12 = pts[:,2:] - pts[:,:2]
        # 从线段起点到障碍物中心的向量
        e1x = self.x - pts[:,:2]
        
        """ test """
        if is_tested:
            print("\n e12 shape: \n",e12.shape)
            print("\n e12:\n",e12)
            print("\n e1x:\n",e1x)
            print("\n e12*e1x shape :",(e12 * e1x).shape)

        # 计算线段上距离障碍物中心最近点的参数λ（clip函数限制大小在0-1之间）
        e12_dot_e1x = np.sum(e12 * e1x, axis=1) # 压缩了列维度
        e12_dot_e12 = np.sum(e12**2, axis=1) # 压缩了列维度
        lam = np.clip(e12_dot_e1x / e12_dot_e12, 0, 1)

        """ test """
        if is_tested:
            print("\n e12_dot_e1x shape: \n",e12_dot_e1x.shape )
            print("\n e12_dot_e1x: \n",e12_dot_e1x )
            print("\n e12_dot_e12 shape: \n",e12_dot_e12.shape )
            print("\n e12_dot_e12: \n",e12_dot_e12 )
            print("\n lam: \n",lam )
            print("\n lam shape: \n",lam.shape )
            e12T = e12.T  # 转置以便广播运算
            lam_new_axis = lam[:, np.newaxis]  # 增加新轴以便广播运算
            print("\n lam_new_axis: \n",lam_new_axis)
            print("\n lam_new_axis shape: \n",lam_new_axis.shape )
            print("\n e12T: \n",e12T)
            print("\n lam_new_axis * e12: \n",lam_new_axis * e12)
            print("\n lam*e12T: \n",lam*e12T)
            print("\n (lam*e12T).T : \n",(lam*e12T).T)


        # 计算线段上最近点到障碍物中心的距离平方
        lam = lam[:,np.newaxis]
        d_sq = np.sum((e1x - lam * e12)**2, axis=1)
        # 如果距离小于障碍物半径，则发生碰撞
        return d_sq < self.r_sq
    
    def isRobotCollision(self, pts):
        """
        检查机械臂当前配置是否与障碍物发生碰撞
        参数：
        pts: (n, 2)数组，机械臂运动学链上每个点的二维坐标
        返回：布尔值，表示是否发生碰撞
        """
        # 将机械臂关节连接线段格式化
        pts_ = np.empty((pts.shape[0]-1, 4)) # 创建一个形状为 (n-1, 4) 的空数组
        # np的切片操作，区间左闭右开
        pts_[:,:2] = pts[:-1]  # 线段起点
        pts_[:,2:] = pts[1:]   # 线段终点
        # np.any 用于判断数组中是否存在至少一个元素为 True。如果存在，则返回 True，否则返回 False
        return np.any(self.isCollision(pts_))
        
    def isTrajectoryCollision(self, pts):
        """
        检查轨迹是否与障碍物发生碰撞
        参数：
        pts: (k, n, 2)数组，k是时间步数，n是机械臂关节数，2是二维坐标
             每个矩阵代表机械臂的运动学链，每行是机械臂链上的一个二维点
        返回：长度为k的布尔数组，每个元素表示该时间步是否发生碰撞
        """
        # 将机械臂链段重构为线段格式进行碰撞检测
        pts_ = np.empty((pts.shape[0] * (pts.shape[1]-1), 4))
        pts_[:,:2] = pts[:,:-1].reshape(-1, 2)  # 线段起点
        pts_[:,2:] = pts[:,1:].reshape(-1, 2)   # 线段终点
        # 检测每个时间步的碰撞情况
        collisions_over_time = np.any(self.isCollision(pts_).reshape(pts.shape[0], pts.shape[1]-1), axis=1)
        return collisions_over_time
    

# 创建碰撞环境实例并测试碰撞检测
# env = CollisionEnvironment()  # 创建碰撞环境实例
# mp = Manipulator()  # 创建机械臂实例
# q = np.array([np.pi/4, 0., 0.])  # 定义关节角度
# positions = mp.fk(q)  # 计算正向运动学，获取各关节位置
# print("\n 关节坐标:\n", positions)  # 打印关节位置

# # 画出关节位置和障碍物
# fig, ax = plt.subplots()
# ax.plot(positions[:,0], positions[:,1], '-o', label='arm pos')
# # 画出障碍物
# circle = patches.Circle(env.x, env.r, color='r', alpha=0.5, label='obstacle')
# ax.add_patch(circle)
# # 设置工作空间边界
# ax.set_aspect('equal', 'box')
# ax.grid()
# ax.legend()
# plt.title('pos')
# plt.show()

# env.isRobotCollision(positions)

In [32]:
# 绘制机械臂和碰撞环境的函数
def plotEnvironment(ax, env):
    """
    在给定的坐标轴上绘制碰撞环境
    参数：
    ax: matplotlib坐标轴对象
    env: CollisionEnvironment实例
    """
    # 设置坐标轴范围为工作空间边界
    # ax.set_xlim(env.x_min[0], env.x_max[0])
    # ax.set_ylim(env.x_min[1], env.x_max[1])
    ax.set_aspect('equal')  # 保持坐标轴比例一致
    # 添加红色半透明的圆形障碍物
    ax.add_patch(patches.Circle(env.x, env.r, facecolor='r', edgecolor='None', alpha=0.5))

def plotRobot(ax, robot, q, color='k'):
    """
    在给定坐标轴上绘制机械臂
    参数：
    ax: matplotlib坐标轴对象
    robot: Manipulator实例
    q: 关节角度数组
    color: 绘制颜色，默认为黑色
    """
    # 通过正向运动学计算机械臂各关节位置
    X = robot.fk(q)
    # 绘制机械臂链接线（黑色线条）
    ax.plot(X[:,0], X[:,1], 'k')
    # 绘制关节点（彩色圆圈，除基座外）
    ax.plot(X[1:,0], X[1:,1], color+'o', markersize=6)
    # 绘制所有关节点（黑色小圆圈）
    ax.plot(X[:,0], X[:,1], 'k*', markersize=6)

""" 测试绘图函数 """
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# # 创建一个新的图形
# fig, ax = plt.subplots()

# # 创建机械臂和碰撞环境实例
# robot = Manipulator()           # 创建2D机械臂对象
# env = CollisionEnvironment()    # 创建碰撞环境对象

# # 绘制环境
# plotEnvironment(ax, env)

# # 绘制机械臂
# q = [np.pi/2, 0, 0]
# plotRobot(ax, robot, q, 'b')

# # 显示图形
# plt.show()

' 测试绘图函数 '

In [ ]:
# 创建机械臂和碰撞环境实例
robot = Manipulator()           # 创建2D机械臂对象
env = CollisionEnvironment()    # 创建碰撞环境对象

# 定义初始和目标关节角度
q0 = np.array([4*np.pi/8, -7*np.pi/8, 0])  # 初始关节角度（起始配置）
qd = np.array([3*np.pi/8, -5*np.pi/8, 0])  # 目标关节角度（目标配置）
q_init = 0.5 * (q0 + qd)                # 初始猜测的中间配置（起始和目标的平均值）



In [ ]:
# 在配置空间中采样关节角度并检查是否无碰撞
# 这一步用于生成配置空间的可行性地图

# 设置采样分辨率
n1_samples = 100  # 第一关节的采样点数
n2_samples = 100  # 第二关节的采样点数
n3_samples = 100  # 第三关节的采样点数

# 在关节限制范围内生成均匀采样点
q1 = np.linspace(robot.q_min[0], robot.q_max[0], n1_samples)  # 第一关节角度采样
q2 = np.linspace(robot.q_min[1], robot.q_max[1], n2_samples)  # 第二关节角度采样
q3 = np.linspace(robot.q_min[2], robot.q_max[2], n3_samples)  # 第三关节角度采样

# 初始化配置空间可行性矩阵（1表示可行，0表示不可行）
c_space = np.zeros((n1_samples, n2_samples,n3_samples))

# 遍历所有关节角度组合，检查可行性
for i in range(n1_samples):
    for j in range(n2_samples):
        for k in range(n3_samples):
          q = np.array([q1[i], q2[j], q3[k]])  # 当前关节配置
          X = robot.fk(q)               # 计算对应的笛卡尔空间位置
          
          # 检查关节限制约束
          if np.any(q <= robot.q_min) or np.any(q >= robot.q_max):
              c_space[j,i] = 0  # 超出关节限制，标记为不可行
          # 检查工作空间边界约束
          elif np.any(X[:,0] < env.x_min[0]) or np.any(X[:,0] > env.x_max[0]) or np.any(X[:,1] < env.x_min[1]) or np.any(X[:,1] > env.x_max[1]):
              c_space[j,i] = 0  # 超出工作空间，标记为不可行
          # 检查碰撞约束
          elif env.isRobotCollision(robot.fk(q)):
              c_space[j,i] = 0  # 发生碰撞，标记为不可行
          else:
              c_space[j,i] = 1  # 所有约束都满足，标记为可行

